In [ ]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - old folder structure
# Gabrielle Baxter 01/29/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np

all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
subjects.sort()

for subject in subjects[15:]:
    # subject = 'WCMyofascial2036'
    maps_folders = ['maps']
    parent_folder = os.path.join(all_folders,subject)
    parent_nifti_folder = os.path.join(parent_folder,'derivatives/all')

    for folder in maps_folders:
        maps_folder = os.path.join(parent_folder,folder)
        print(maps_folder)

        # Get diffusion times
        # diffusion_times = [s[0:6] for s in os.listdir(maps_folder)] 
        # diffusion_times = list(set(diffusion_times))
        # diffusion_times.sort()
        diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
        print(diffusion_times)

        # Get names of maps
        # maps = [s.split("_")[-1][:-7] for s in os.listdir(maps_folder)] # Hopefully this works as long as maps have naming convention e.g. 0022ms_AD.nii.gz or 0300ms_colFA.nii.gz
        # maps = [s for s in maps if 'reg' not in s]
        # maps = list(set(maps))
        # maps.sort()

        # Fixed is lowest diffusion time- assume you've drawn an ROI on this diffusion time
        if "rescan" in parent_folder:
            initial_scan_parent_folder = parent_folder[:-7]
            initial_scan_nifti_folder = os.path.join(initial_scan_parent_folder,'derivatives/all')
            print('Fixed',os.path.join(initial_scan_nifti_folder,diffusion_times[0] + '/dwiec.nii'))
            fixed = ants.image_read(os.path.join(initial_scan_nifti_folder,diffusion_times[0] + '/dwiec.nii'))
        else:
            fixed = ants.image_read(os.path.join(parent_nifti_folder,diffusion_times[0] + '/dwiec.nii'))
            print('Fixed',os.path.join(parent_nifti_folder,diffusion_times[0] + '/dwiec.nii'))
        fixed_array = fixed.numpy()
        fixed_b0= fixed_array[:,:,:,0]
        fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

        for dt in diffusion_times[1:]: #iterate through the other diffusion times registering to lowest
            print('Moving',os.path.join(parent_nifti_folder, dt + '/dwiec.nii'))
            moving = ants.image_read(os.path.join(parent_nifti_folder, dt + '/dwiec.nii'))
            moving_array = moving.numpy()
            nx,ny,nz,nvols = np.shape(moving_array)

            # # Register dwi - not really necessary
            # registered_dwi = np.zeros([nx,ny,nz,nvols])
            # save_filename = os.path.join(parent_nifti_folder, dt + '/dwiec_reg.nii')
            # if not os.path.exists(save_filename):
            #     # register dwi
            #     for vol in range(0,nvols): # register all the volumes of the dwi for each diffusion time to the b0 of the lowest diffusion time image
            #         moving_temp = moving_array[:,:,:,vol]
            #         moving_ants = ants.from_numpy(moving_temp, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            #         reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='SyN')  
            #         warped_image = ants.apply_transforms(
            #             fixed=fixed_ants,
            #             moving=moving_ants,
            #             transformlist=reg['fwdtransforms'],
            #             interpolator='bSpline'  
            #         )
            #         warped_image_array = warped_image.numpy()
            #         registered_dwi[:,:,:,vol] = warped_image_array

            #     registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
            #     registered_dwi_ants.to_filename(save_filename)

            # Register maps
            moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
            moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='SyN')    
            
            maps = ['AD','RD','FA','L2','L3']
            for map in maps:
                current_map = ants.image_read(os.path.join(maps_folder,dt + '_' + map + '.nii.gz'))
                map_array = current_map.numpy()
                print(os.path.join(maps_folder,dt + '_' + map + '.nii.gz'))

                warped_map = ants.apply_transforms(
                    fixed=fixed_ants,
                    moving=current_map,
                    transformlist=reg['fwdtransforms'],
                    interpolator='bSpline'  
                )
                save_filename = os.path.join(maps_folder,dt + '_' + map + '_withinscanreg.nii.gz')
                warped_map.to_filename(save_filename)


/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps
['0022ms', '0042ms', '0081ms', '0156ms', '0300ms']
Fixed /Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/derivatives/all/0022ms/dwiec.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/derivatives/all/0042ms/dwiec.nii
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps/0042ms_AD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps/0042ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps/0042ms_FA.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps/0042ms_L2.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps/0042ms_L3.nii.gz
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/derivatives/all/0081ms/dwiec.nii
/Users/gabriellebaxter/Documents/HEAL_Volunteers/WCMyofascial2016/maps/0081ms_AD.nii.gz
/Users/gabriellebaxter/Documents/

In [2]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - batch
# Gabrielle Baxter 04/13/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np
from tqdm import tqdm

# all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
# subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
parent_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/maps_smoothed'
dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi'
all_maps = os.listdir(parent_folder)
all_maps = [s for s in all_maps if 'WCMyofascial' in s]
subjects = [s[0:16] for s in all_maps]
subjects = list(set(subjects))
subjects.sort()

for subject in tqdm(subjects[55:]):
    map_files = [s for s in all_maps if subject in s]
    print(map_files)

    diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']

    fixed = ants.image_read(os.path.join(dwi_folder,subject + '_' + diffusion_times[0] + '.nii'))
    fixed_array = fixed.numpy()
    fixed_b0= fixed_array[:,:,:,0]
    fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

    for dt in diffusion_times[1:]: #iterate through the other diffusion times registering to lowest
        print('Moving',os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
        moving = ants.image_read(os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
        moving_array = moving.numpy()
        nx,ny,nz,nvols = np.shape(moving_array)

        # Register maps
        moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
        moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
        reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='SyN')    

        # registered_dwi = np.zeros([nx,ny,nz,nvols])
        # # register full dwi 
        # for vol in range(0,nvols):
        #     temp_array = moving_array[:,:,:,vol]
        #     temp_ants = ants.from_numpy(temp_array, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
        #     warped_image = ants.apply_transforms(fixed=fixed_ants,moving=temp_ants,transformlist=reg['fwdtransforms'],interpolator='bSpline')
        #     warped_image_array = warped_image.numpy()
        #     registered_dwi[:,:,:,vol] = warped_image_array
        # registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
        # save_filename = os.path.join(dwi_folder,subject + '_' + dt + '_reg.nii')
        # registered_dwi_ants.to_filename(save_filename)
        
        # maps = ['AD','RD','FA','L2','L3']
        maps = ['MD']
        for map in maps:
            current_map = ants.image_read(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            map_array = current_map.numpy()
            print(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

            warped_map = ants.apply_transforms(
                fixed=fixed_ants,
                moving=current_map,
                transformlist=reg['fwdtransforms'],
                interpolator='bSpline'  
            )
            save_filename = os.path.join(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
            warped_map.to_filename(save_filename)

        # maps = ['eig']
        # for map in maps:
        #     current_map = ants.image_read(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
        #     map_array = current_map.numpy()
        #     print(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

        #     warped_map = ants.apply_transforms(
        #         fixed=fixed_ants,
        #         moving=current_map,
        #         transformlist=reg['fwdtransforms'],
        #         interpolator='bSpline',
        #         imagetype=1  
        #     )
        #     save_filename = os.path.join(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
        #     warped_map.to_filename(save_filename)


  0%|          | 0/30 [00:00<?, ?it/s]

['WCMyofascial2060_0042ms_RD_withinscanreg.nii.gz', 'WCMyofascial2060_0300ms_AD_withinscanreg.nii.gz', 'WCMyofascial2060_0300ms_AD.nii.gz', 'WCMyofascial2060_0156ms_AD_right_temporalis.nii.gz', 'WCMyofascial2060_0156ms_FA.nii.gz', 'WCMyofascial2060_0300ms_FA_right_masseter.nii.gz', 'WCMyofascial2060_0081ms_AD_right_masseter.nii.gz', 'WCMyofascial2060_0022ms_AD.nii.gz', 'WCMyofascial2060_0081ms_AD.nii.gz', 'WCMyofascial2060_0300ms_RD_withinscanreg.nii.gz', 'WCMyofascial2060_0042ms_AD_withinscanreg.nii.gz', 'WCMyofascial2060_0156ms_FA_right_temporalis.nii.gz', 'WCMyofascial2060_0156ms_FA_left_masseter.nii.gz', 'WCMyofascial2060_0042ms_AD.nii.gz', 'WCMyofascial2060_0042ms_RD_right_temporalis.nii.gz', 'WCMyofascial2060_0081ms_FA_left_masseter.nii.gz', 'WCMyofascial2060_0081ms_FA_right_temporalis.nii.gz', 'WCMyofascial2060_0300ms_FA_right_temporalis.nii.gz', 'WCMyofascial2060_0300ms_AD_left_masseter.nii.gz', 'WCMyofascial2060_0042ms_RD_left_masseter.nii.gz', 'WCMyofascial2060_0300ms_AD_left

  3%|▎         | 1/30 [00:03<01:45,  3.64s/it]

/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/maps_smoothed/WCMyofascial2060_0300ms_MD.nii.gz
['WCMyofascial2061_0300ms_FA_right_temporalis.nii.gz', 'WCMyofascial2061_0081ms_FA_right_temporalis.nii.gz', 'WCMyofascial2061_0042ms_FA_left_temporalis.nii.gz', 'WCMyofascial2061_0042ms_AD_left_temporalis.nii.gz', 'WCMyofascial2061_0042ms_RD_left_masseter.nii.gz', 'WCMyofascial2061_0042ms_RD_right_masseter.nii.gz', 'WCMyofascial2061_0300ms_AD_left_masseter.nii.gz', 'WCMyofascial2061_0081ms_FA_right_masseter.nii.gz', 'WCMyofascial2061_0042ms_FA.nii.gz', 'WCMyofascial2061_0300ms_AD_right_masseter.nii.gz', 'WCMyofascial2061_0156ms_RD_right_masseter.nii.gz', 'WCMyofascial2061_0081ms_FA_left_temporalis.nii.gz', 'WCMyofascial2061_0156ms_RD_left_temporalis.nii.gz', 'WCMyofascial2061_0081ms_AD_left_temporalis.nii.gz', 'WCMyofascial2061_0300ms_RD_left_temporalis.nii.gz', 'WCMyofascial2061_0300ms_RD_left_masseter.nii.gz', 'WCMyofascial2061_0081ms_AD_right_temporalis.nii.gz', 'WCMyofascial206

  7%|▋         | 2/30 [00:07<01:39,  3.56s/it]

...

/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/maps_smoothed/WCMyofascial2077_0300ms_MD.nii.gz
['WCMyofascial2078_0300ms_MD.nii.gz', 'WCMyofascial2078_0300ms_AD_left_masseter.nii.gz', 'WCMyofascial2078_0022ms_MD.nii.gz', 'WCMyofascial2078_0156ms_FA_right_masseter.nii.gz', 'WCMyofascial2078_0042ms_RD_left_masseter.nii.gz', 'WCMyofascial2078_0081ms_MD.nii.gz', 'WCMyofascial2078_0042ms_RD_left_temporalis.nii.gz', 'WCMyofascial2078_0081ms_RD_right_masseter.nii.gz', 'WCMyofascial2078_0156ms_FA_withinscanreg.nii.gz', 'WCMyofascial2078_0081ms_FA_withinscanreg.nii.gz', 'WCMyofascial2078_0081ms_RD_right_temporalis.nii.gz', 'WCMyofascial2078_0300ms_RD_right_temporalis.nii.gz', 'WCMyofascial2078_0156ms_FA_left_temporalis.nii.gz', 'WCMyofascial2078_0081ms_RD_left_temporalis.nii.gz', 'WCMyofascial2078_0042ms_AD_left_masseter.nii.gz', 'WCMyofascial2078_0300ms_RD_left_masseter.nii.gz', 'WCMyofascial2078_0156ms_AD_left_temporalis.nii.gz', 'WCMyofascial2078_0300ms_AD_left_temporalis.nii.gz', 

 60%|██████    | 18/30 [01:03<00:42,  3.55s/it]

/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/maps_smoothed/WCMyofascial2078_0300ms_MD.nii.gz
['WCMyofascial2079_0156ms_RD_right_temporalis.nii.gz', 'WCMyofascial2079_0300ms_RD.nii.gz', 'WCMyofascial2079_0042ms_FA_right_temporalis.nii.gz', 'WCMyofascial2079_0022ms_RD.nii.gz', 'WCMyofascial2079_0156ms_AD_right_masseter.nii.gz', 'WCMyofascial2079_0300ms_AD_withinscanreg.nii.gz', 'WCMyofascial2079_0042ms_RD_withinscanreg.nii.gz', 'WCMyofascial2079_0081ms_RD.nii.gz', 'WCMyofascial2079_0300ms_RD_right_masseter.nii.gz', 'WCMyofascial2079_0081ms_FA_left_masseter.nii.gz', 'WCMyofascial2079_0156ms_FA_left_masseter.nii.gz', 'WCMyofascial2079_0042ms_RD.nii.gz', 'WCMyofascial2079_0042ms_AD_right_masseter.nii.gz', 'WCMyofascial2079_0042ms_AD_withinscanreg.nii.gz', 'WCMyofascial2079_0156ms_MD.nii.gz', 'WCMyofascial2079_0300ms_RD_withinscanreg.nii.gz', 'WCMyofascial2079_0042ms_AD_right_temporalis.nii.gz', 'WCMyofascial2079_0300ms_MD.nii.gz', 'WCMyofascial2079_0081ms_AD_right_masseter.nii.g

 60%|██████    | 18/30 [01:05<00:43,  3.62s/it]


KeyboardInterrupt: 

In [ ]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - batch USING ROIS
# Gabrielle Baxter 04/13/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np

# all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
# subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
parent_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/maps_smoothed'
dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi_smoothed'
all_maps = os.listdir(parent_folder)
subjects = [s[0:16] for s in all_maps]
subjects = list(set(subjects))
subjects.sort()

for subject in subjects:
    map_files = [s for s in all_maps if subject in s]
    print(map_files)

    diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']

    fixed = ants.image_read(os.path.join(dwi_folder,subject + '_' + diffusion_times[0] + '.nii'))
    fixed_array = fixed.numpy()
    fixed_b0= fixed_array[:,:,:,0]
    fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

    for dt in diffusion_times[1:]: #iterate through the other diffusion times registering to lowest
        print('Moving',os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
        moving = ants.image_read(os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
        moving_array = moving.numpy()
        nx,ny,nz,nvols = np.shape(moving_array)

        # Register maps
        moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
        moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
        reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='SyN')    

        # registered_dwi = np.zeros([nx,ny,nz,nvols])
        # # register full dwi 
        # for vol in range(0,nvols):
        #     temp_array = moving_array[:,:,:,vol]
        #     temp_ants = ants.from_numpy(temp_array, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
        #     warped_image = ants.apply_transforms(fixed=fixed_ants,moving=temp_ants,transformlist=reg['fwdtransforms'],interpolator='bSpline')
        #     warped_image_array = warped_image.numpy()
        #     registered_dwi[:,:,:,vol] = warped_image_array
        # registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
        # save_filename = os.path.join(dwi_folder,subject + '_' + dt + '_reg.nii')
        # registered_dwi_ants.to_filename(save_filename)
        
        # maps = ['AD','RD','FA','L2','L3']
        # for map in maps:
        #     current_map = ants.image_read(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
        #     map_array = current_map.numpy()
        #     print(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

        #     warped_map = ants.apply_transforms(
        #         fixed=fixed_ants,
        #         moving=current_map,
        #         transformlist=reg['fwdtransforms'],
        #         interpolator='bSpline'  
        #     )
        #     save_filename = os.path.join(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
        #     warped_map.to_filename(save_filename)

        # maps = ['eig']
        # for map in maps:
        #     current_map = ants.image_read(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
        #     map_array = current_map.numpy()
        #     print(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

        #     warped_map = ants.apply_transforms(
        #         fixed=fixed_ants,
        #         moving=current_map,
        #         transformlist=reg['fwdtransforms'],
        #         interpolator='bSpline',
        #         imagetype=1  
        #     )
        #     save_filename = os.path.join(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
        #     warped_map.to_filename(save_filename)


In [5]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - batch USING ROIS - USE THIS!
# Gabrielle Baxter 05/18/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np
from tqdm import tqdm

# all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
# subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
maps_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/maps_smoothed'
dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi_smoothed'
nonsmoothed_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi'
roi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/roi_resampled'
all_maps = os.listdir(maps_folder)
subjects = [s[0:16] for s in os.listdir(roi_folder)]
subjects = list(set(subjects))
subjects.sort()

diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
rois = ['left_masseter','right_masseter','left_temporalis','right_temporalis']

for subject in tqdm(subjects[54:]):
    # Load ROI
    roi_file = os.path.join(roi_folder,subject + '.nii.gz')
    roi_ants = ants.image_read(roi_file)
    roi = roi_ants.numpy()

    # Get correpsonding map files
    map_files = [s for s in all_maps if subject in s]
    print(map_files)

    # Get fixed lowest diffusion time b0 to register everything to
    fixed = ants.image_read(os.path.join(dwi_folder,subject + '_' + diffusion_times[0] + '.nii'))
    fixed_array = fixed.numpy()
    fixed_b0= fixed_array[:,:,:,0]
    fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

    for dt in diffusion_times[1:]: #iterate through the other diffusion times registering to lowest
        print('Moving',os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
        moving = ants.image_read(os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
        moving_array = moving.numpy()
        nx,ny,nz,nvols = np.shape(moving_array)

        # Get b0 of the diffusion time and get registration
        moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
        moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
        reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
        warped_vol = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg['fwdtransforms'],interpolator='linear') # warp first vol for similarity measures

        # # register full dwi - need this for fasciculation measurements
        # registered_dwi = np.zeros([nx,ny,nz,nvols])
        # for vol in range(0,nvols):
        #     temp_array = moving_array[:,:,:,vol]
        #     temp_ants = ants.from_numpy(temp_array, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
        #     warped_image = ants.apply_transforms(fixed=fixed_ants,moving=temp_ants,transformlist=reg['fwdtransforms'],interpolator='linear')
        #     warped_image_array = warped_image.numpy()
        #     registered_dwi[:,:,:,vol] = warped_image_array
        # registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
        # save_filename = os.path.join(dwi_folder,subject + '_' + dt + '_reg.nii')
        # registered_dwi_ants.to_filename(save_filename)
        
        # Register maps using global reg
        # maps = ['AD','RD','FA']
        maps = ['MD']
        for map in maps:
            current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            map_array = current_map.numpy()
            print(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

            warped_map = ants.apply_transforms(
                fixed=fixed_ants,
                moving=current_map,
                transformlist=reg['fwdtransforms'],
                interpolator='bSpline'  
            )
            save_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
            warped_map.to_filename(save_filename)

        # Now iterate through ROIs to refine
        for label in range(1,5):
            temp_roi = (roi == label)
            temp_roi_name = rois[label-1]
            print(temp_roi_name)

            coords = np.argwhere(temp_roi) # where is there ROI
            pad = 5
            mins = np.maximum(coords.min(0) - pad, 0)
            maxs = np.minimum(coords.max(0) + pad + 1, temp_roi.shape)

            # create bounding box mask
            bbox_mask = np.zeros_like(temp_roi, dtype=np.uint8)
            bbox_mask[mins[0]:maxs[0], mins[1]:maxs[1], mins[2]:maxs[2]] = 1

            bbox_mask_ants = ants.from_numpy(bbox_mask.astype(np.uint8),origin=fixed_ants.origin,spacing=fixed_ants.spacing,direction=fixed_ants.direction)
            similarity_measure = 'MattesMutualInformation'
            similarity_val = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=moving_ants*bbox_mask_ants,metric_type=similarity_measure) # calculate similarity just in bounding box before warp
            print(dt,"Initial Global:", similarity_val) # before first warp
            similarity_val_initial = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=warped_vol*bbox_mask_ants,metric_type=similarity_measure) # calculate similarity just in bounding box
            print(dt,"Global after affine:", similarity_val_initial) # after first warp

            # Second reg- rigid just in bounding box around ROI
            reg_roi = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Rigid',initial_transform=reg['fwdtransforms'], mask=bbox_mask_ants,random_seed=42,mask_all_stages=True) #includes intial reg
            warped_vol_roi = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg_roi['fwdtransforms'],interpolator='linear') # roi warp
            similarity_val_roi = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=warped_vol_roi*bbox_mask_ants,metric_type=similarity_measure)
            print(dt,"After ROI warp:", similarity_val_roi)

            # If roi warp makes it worse, just use the global one
            if abs(similarity_val_roi) < abs(similarity_val_initial):
                final_reg = reg
            else:
                final_reg = reg_roi

            # apply to maps
            # maps = ['AD','RD','FA']
            maps = ['MD']
            for map in maps:
                current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
                map_array = current_map.numpy()

                warped_map = ants.apply_transforms(
                    fixed=fixed_ants,
                    moving=current_map,
                    transformlist=final_reg['fwdtransforms'],
                    interpolator='linear'  
                )

                save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
                warped_map.to_filename(save_filename)

        # maps = ['eig']
        # for map in maps:
        #     current_map = ants.image_read(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
        #     map_array = current_map.numpy()
        #     print(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

        #     warped_map = ants.apply_transforms(
        #         fixed=fixed_ants,
        #         moving=current_map,
        #         transformlist=reg['fwdtransforms'],
        #         interpolator='bSpline',
        #         imagetype=1  
        #     )
        #     save_filename = os.path.join(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
        #     warped_map.to_filename(save_filename)


  0%|          | 0/31 [00:00<?, ?it/s]

['WCMyofascial2059_0156ms_MD_right_temporalis.nii.gz', 'WCMyofascial2059_0300ms_FA_left_masseter.nii.gz', 'WCMyofascial2059_0300ms_MD.nii.gz', 'WCMyofascial2059_0156ms_RD_withinscanreg.nii.gz', 'WCMyofascial2059_0081ms_MD.nii.gz', 'WCMyofascial2059_0156ms_AD_right_temporalis.nii.gz', 'WCMyofascial2059_0156ms_FA_right_masseter.nii.gz', 'WCMyofascial2059_0081ms_RD_withinscanreg.nii.gz', 'WCMyofascial2059_0022ms_MD.nii.gz', 'WCMyofascial2059_0081ms_MD_left_temporalis.nii.gz', 'WCMyofascial2059_0042ms_FA_left_masseter.nii.gz', 'WCMyofascial2059_0081ms_RD_right_masseter.nii.gz', 'WCMyofascial2059_0042ms_RD_right_temporalis.nii.gz', 'WCMyofascial2059_0156ms_AD_withinscanreg.nii.gz', 'WCMyofascial2059_0300ms_MD_right_masseter.nii.gz', 'WCMyofascial2059_0156ms_MD_left_masseter.nii.gz', 'WCMyofascial2059_0156ms_FA_right_temporalis.nii.gz', 'WCMyofascial2059_0042ms_FA_right_masseter.nii.gz', 'WCMyofascial2059_0042ms_MD_left_temporalis.nii.gz', 'WCMyofascial2059_0042ms_MD.nii.gz', 'WCMyofascial20

  3%|▎         | 1/31 [00:08<04:04,  8.14s/it]

0300ms After ROI warp: -0.24859148263931274
['WCMyofascial2060_0042ms_RD_withinscanreg.nii.gz', 'WCMyofascial2060_0300ms_AD_withinscanreg.nii.gz', 'WCMyofascial2060_0300ms_AD.nii.gz', 'WCMyofascial2060_0156ms_AD_right_temporalis.nii.gz', 'WCMyofascial2060_0156ms_FA.nii.gz', 'WCMyofascial2060_0300ms_FA_right_masseter.nii.gz', 'WCMyofascial2060_0081ms_AD_right_masseter.nii.gz', 'WCMyofascial2060_0022ms_AD.nii.gz', 'WCMyofascial2060_0081ms_AD.nii.gz', 'WCMyofascial2060_0300ms_RD_withinscanreg.nii.gz', 'WCMyofascial2060_0042ms_AD_withinscanreg.nii.gz', 'WCMyofascial2060_0156ms_FA_right_temporalis.nii.gz', 'WCMyofascial2060_0156ms_FA_left_masseter.nii.gz', 'WCMyofascial2060_0042ms_AD.nii.gz', 'WCMyofascial2060_0042ms_RD_right_temporalis.nii.gz', 'WCMyofascial2060_0081ms_FA_left_masseter.nii.gz', 'WCMyofascial2060_0081ms_FA_right_temporalis.nii.gz', 'WCMyofascial2060_0300ms_FA_right_temporalis.nii.gz', 'WCMyofascial2060_0300ms_AD_left_masseter.nii.gz', 'WCMyofascial2060_0042ms_RD_left_masset

  6%|▋         | 2/31 [00:15<03:48,  7.86s/it]

...

0300ms After ROI warp: -0.37786391377449036
['WCMyofascial2089_0156ms_AD_withinscanreg.nii.gz', 'WCMyofascial2089_0081ms_AD_withinscanreg.nii.gz', 'WCMyofascial2089_0156ms_FA_right_temporalis.nii.gz', 'WCMyofascial2089_0081ms_AD_right_masseter.nii.gz', 'WCMyofascial2089_0042ms_FA_left_masseter.nii.gz', 'WCMyofascial2089_0042ms_RD_right_temporalis.nii.gz', 'WCMyofascial2089_0300ms_FA_right_masseter.nii.gz', 'WCMyofascial2089_0042ms_AD.nii.gz', 'WCMyofascial2089_0156ms_AD_right_temporalis.nii.gz', 'WCMyofascial2089_0156ms_RD_withinscanreg.nii.gz', 'WCMyofascial2089_0300ms_AD.nii.gz', 'WCMyofascial2089_0081ms_RD_withinscanreg.nii.gz', 'WCMyofascial2089_0081ms_AD.nii.gz', 'WCMyofascial2089_0300ms_FA_left_masseter.nii.gz', 'WCMyofascial2089_0022ms_AD.nii.gz', 'WCMyofascial2089_0156ms_FA.nii.gz', 'WCMyofascial2089_0081ms_AD_right_temporalis.nii.gz', 'WCMyofascial2089_0300ms_AD_right_temporalis.nii.gz', 'WCMyofascial2089_0156ms_AD_left_masseter.nii.gz', 'WCMyofascial2089_0300ms_FA.nii.gz', 'W

 97%|█████████▋| 30/31 [04:01<00:07,  7.99s/it]

0300ms After ROI warp: -0.3326548933982849
['WCMyofascial2090_0042ms_FA_left_masseter.nii.gz', 'WCMyofascial2090_0042ms_AD_right_temporalis.nii.gz', 'WCMyofascial2090_0156ms_MD.nii.gz', 'WCMyofascial2090_0156ms_AD_right_masseter.nii.gz', 'WCMyofascial2090_0081ms_AD_withinscanreg.nii.gz', 'WCMyofascial2090_0156ms_AD_withinscanreg.nii.gz', 'WCMyofascial2090_0042ms_RD.nii.gz', 'WCMyofascial2090_0300ms_RD.nii.gz', 'WCMyofascial2090_0156ms_RD_right_temporalis.nii.gz', 'WCMyofascial2090_0300ms_RD_right_masseter.nii.gz', 'WCMyofascial2090_0300ms_FA_left_masseter.nii.gz', 'WCMyofascial2090_0081ms_RD.nii.gz', 'WCMyofascial2090_0081ms_RD_withinscanreg.nii.gz', 'WCMyofascial2090_0022ms_RD.nii.gz', 'WCMyofascial2090_0042ms_AD_right_masseter.nii.gz', 'WCMyofascial2090_0156ms_RD_withinscanreg.nii.gz', 'WCMyofascial2090_0042ms_FA_right_temporalis.nii.gz', 'WCMyofascial2090_0042ms_RD_left_temporalis.nii.gz', 'WCMyofascial2090_0081ms_RD_right_temporalis.nii.gz', 'WCMyofascial2090_0300ms_RD_right_tempor

100%|██████████| 31/31 [04:09<00:00,  8.05s/it]

0300ms After ROI warp: -0.2627180814743042


In [3]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - batch USING ROIS- Cornell to Cornell rescan - USE THIS
# Gabrielle Baxter 05/18/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np
from tqdm import tqdm

# all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
# subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
maps_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell_rescan/maps_smoothed'
dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell_rescan/dwi_smoothed'
original_dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi_smoothed'
dwi_unsmoothed_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell_rescan/dwi'
roi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/roi_resampled'
all_maps = os.listdir(maps_folder)
subjects = [s[0:16] for s in os.listdir(roi_folder)]
subjects = list(set(subjects))
subjects.sort()

diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
rois = ['left_masseter','right_masseter','left_temporalis','right_temporalis']

for subject in tqdm(subjects):
    # Load ROI
    roi_file = os.path.join(roi_folder,subject + '.nii.gz')
    roi_ants = ants.image_read(roi_file)
    roi = roi_ants.numpy()

    # Get correpsonding map files
    map_files = [s for s in all_maps if subject in s]
    print(map_files)

    # Get fixed lowest diffusion time b0 to register everything to
    fixed = ants.image_read(os.path.join(original_dwi_folder,subject + '_' + diffusion_times[0] + '.nii'))
    fixed_array = fixed.numpy()
    fixed_b0= fixed_array[:,:,:,0]
    fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

    if os.path.exists(os.path.join(dwi_folder,subject + '_0022ms.nii')): # there is a repeat scan

        for dt in diffusion_times: #iterate through the other diffusion times registering to lowest
            # print('Moving',os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
            # moving = ants.image_read(os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
            # moving_array = moving.numpy()
            # nx,ny,nz,nvols = np.shape(moving_array)

            # # Get b0 of the diffusion time and get registration
            # moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
            # moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            # reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
            # warped_vol = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg['fwdtransforms'],interpolator='linear') # warp first vol for similarity measures

            print('Moving',os.path.join(dwi_unsmoothed_folder,subject + '_' + dt + '.nii'))
            moving = ants.image_read(os.path.join(dwi_unsmoothed_folder,subject + '_' + dt + '.nii'))
            moving_array = moving.numpy()
            nx,ny,nz,nvols = np.shape(moving_array)

            # Get b0 of the diffusion time and get registration
            moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
            moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
            warped_vol = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg['fwdtransforms'],interpolator='linear') # warp first vol for similarity measures

            # register full dwi - need this for fasciculation measurements
            registered_dwi = np.zeros([nx,ny,nz,nvols])
            for vol in range(0,nvols):
                temp_array = moving_array[:,:,:,vol]
                temp_ants = ants.from_numpy(temp_array, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
                warped_image = ants.apply_transforms(fixed=fixed_ants,moving=temp_ants,transformlist=reg['fwdtransforms'],interpolator='linear')
                warped_image_array = warped_image.numpy()
                registered_dwi[:,:,:,vol] = warped_image_array
            registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
            save_filename = os.path.join(dwi_unsmoothed_folder,subject + '_' + dt + '_reg.nii')
            registered_dwi_ants.to_filename(save_filename)
            
            # Register maps using global reg
            # maps = ['AD','RD','FA','L2','L3']
            # maps = ['AD','RD','FA']
            # maps = ['MD']
            # for map in maps:
            #     current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            #     map_array = current_map.numpy()
            #     print(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

            #     warped_map = ants.apply_transforms(
            #         fixed=fixed_ants,
            #         moving=current_map,
            #         transformlist=reg['fwdtransforms'],
            #         interpolator='bSpline'  
            #     )
            #     save_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
            #     warped_map.to_filename(save_filename)

            # # Now iterate through ROIs to refine
            # for label in range(1,5):
            #     temp_roi = (roi == label)
            #     temp_roi_name = rois[label-1]
            #     print(temp_roi_name)

            #     coords = np.argwhere(temp_roi) # where is there ROI
            #     pad = 5
            #     mins = np.maximum(coords.min(0) - pad, 0)
            #     maxs = np.minimum(coords.max(0) + pad + 1, temp_roi.shape)

            #     # create bounding box mask
            #     bbox_mask = np.zeros_like(temp_roi, dtype=np.uint8)
            #     bbox_mask[mins[0]:maxs[0], mins[1]:maxs[1], mins[2]:maxs[2]] = 1

            #     bbox_mask_ants = ants.from_numpy(bbox_mask.astype(np.uint8),origin=fixed_ants.origin,spacing=fixed_ants.spacing,direction=fixed_ants.direction)
            #     similarity_measure = 'MattesMutualInformation'
            #     similarity_val = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=moving_ants*bbox_mask_ants,metric_type=similarity_measure) # calculate similarity just in bounding box before warp
            #     print(dt,"Initial Global:", similarity_val) # before first warp
            #     similarity_val_initial = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=warped_vol*bbox_mask_ants,metric_type=similarity_measure) # calculate similarity just in bounding box
            #     print(dt,"Global after affine:", similarity_val_initial) # after first warp

            #     # Second reg- rigid just in bounding box around ROI
            #     reg_roi = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Rigid',initial_transform=reg['fwdtransforms'], mask=bbox_mask_ants,random_seed=42,mask_all_stages=True) #includes intial reg
            #     warped_vol_roi = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg_roi['fwdtransforms'],interpolator='linear') # roi warp
            #     similarity_val_roi = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=warped_vol_roi*bbox_mask_ants,metric_type=similarity_measure)
            #     print(dt,"After ROI warp:", similarity_val_roi)

            #     # apply to maps
            #     # maps = ['AD','RD','FA','L2','L3']
            #     # maps = ['AD','RD']
            #     maps = ['MD']
            #     for map in maps:
            #         current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            #         map_array = current_map.numpy()

            #         if abs(similarity_val_roi) < abs(similarity_val_initial): # it's worse, so just use original warp
            #             global_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
            #             warped_map = ants.image_read(global_filename)
            #             save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
            #             warped_map.to_filename(save_filename)

            #         else:
            #             warped_map = ants.apply_transforms(
            #                 fixed=fixed_ants,
            #                 moving=current_map,
            #                 transformlist=reg_roi['fwdtransforms'],
            #                 interpolator='bSpline'  
            #             )
            #             save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
            #             warped_map.to_filename(save_filename)

            # if dt == '0022ms':
            #     # Register sigmas
            #     maps = ['sigma']
            #     sigma_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell_rescan/sigma'
            #     for map in maps:
            #         current_map = ants.image_read(os.path.join(sigma_folder,subject + '_sigma.nii'))
            #         map_array = current_map.numpy()
            #         print(os.path.join(sigma_folder,subject + '_sigma.nii'))

            #         warped_map = ants.apply_transforms(
            #             fixed=fixed_ants,
            #             moving=current_map,
            #             transformlist=reg['fwdtransforms'],
            #             interpolator='bSpline'  
            #         )
            #         save_filename = os.path.join(sigma_folder,subject + '_sigma_reg2wcmscan1.nii')
            #         # save_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
            #         warped_map.to_filename(save_filename)

  2%|▏         | 2/85 [00:00<00:06, 12.68it/s]

[]
[]
[]


  5%|▍         | 4/85 [00:00<00:06, 12.89it/s]

[]
[]
[]


  9%|▉         | 8/85 [00:00<00:05, 12.95it/s]

...

 72%|███████▏  | 61/85 [01:42<00:59,  2.49s/it]

['WCMyofascial2067_0022ms_MD.nii.gz', 'WCMyofascial2067_0022ms_RD_reg2wcmscan1.nii.gz', 'WCMyofascial2067_0022ms_FA_reg2wcmscan1.nii.gz', 'WCMyofascial2067_0022ms_AD_right_masseter.nii.gz', 'WCMyofascial2067_0022ms_MD_left_masseter.nii.gz', 'WCMyofascial2067_0022ms_MD_reg2wcmscan1.nii.gz', 'WCMyofascial2067_0022ms_FA_right_temporalis.nii.gz', 'WCMyofascial2067_0022ms_RD.nii.gz', 'WCMyofascial2067_0022ms_MD_right_masseter.nii.gz', 'WCMyofascial2067_0022ms_RD_left_masseter.nii.gz', 'WCMyofascial2067_0022ms_AD_right_temporalis.nii.gz', 'WCMyofascial2067_0022ms_RD_left_temporalis.nii.gz', 'WCMyofascial2067_0022ms_AD_left_masseter.nii.gz', 'WCMyofascial2067_0022ms_MD_right_temporalis.nii.gz', 'WCMyofascial2067_0022ms_FA_left_masseter.nii.gz', 'WCMyofascial2067_0022ms_FA.nii.gz', 'WCMyofascial2067_0022ms_RD_right_masseter.nii.gz', 'WCMyofascial2067_0022ms_AD_left_temporalis.nii.gz', 'WCMyofascial2067_0022ms_FA_left_temporalis.nii.gz', 'WCMyofascial2067_0022ms_AD_reg2wcmscan1.nii.gz', 'WCMyof

 72%|███████▏  | 61/85 [01:42<00:40,  1.69s/it]

Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell_rescan/dwi/WCMyofascial2067_0042ms.nii


ValueError: File /Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell_rescan/dwi/WCMyofascial2067_0042ms.nii does not exist!

In [ ]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - batch USING ROIS- Cornell to NYU rescan - DONT USE THIS
# Gabrielle Baxter 05/18/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np
import pandas as pd

# all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
# subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
maps_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed'
dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi_smoothed'
original_dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi_smoothed'
nonsmoothed_dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi'
roi_folder = '/Users/gabriellebaxter/Documents/segmentation/Dataset001_HEALv1/roi_resampled'
all_maps = os.listdir(maps_folder)
cornell_subjects = [s[0:16] for s in os.listdir(roi_folder)]
cornell_subjects = list(set(cornell_subjects))
cornell_subjects.sort()

diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
rois = ['left_masseter','right_masseter','left_temporalis','right_temporalis']

column_names = ['WCMID','NYUID','ROI','initial_mmi','after_global_mmi','after_roi_mmi']
df = pd.DataFrame(columns = column_names)
csv_filename = 'nyu_mmi.csv'

for cornell_subject in cornell_subjects:
    # Load ROI
    roi_file = os.path.join(roi_folder,cornell_subject + '.nii.gz')
    roi_ants = ants.image_read(roi_file)
    roi = roi_ants.numpy()

    # Get correpsonding map files
    map_files = [s for s in all_maps if cornell_subject in s]
    # print(map_files)

    # Get fixed lowest diffusion time b0 to register everything to
    fixed = ants.image_read(os.path.join(original_dwi_folder,cornell_subject + '_' + diffusion_times[0] + '.nii'))
    fixed_array = fixed.numpy()
    fixed_b0= fixed_array[:,:,:,0]
    fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

    matching_codes = pd.read_csv('/Users/gabriellebaxter/Documents/code/HEAL_scripts/rescan_rpbm/matching_codes.csv')
    temp_df = matching_codes[matching_codes['WCM_ID'] == int(cornell_subject[12:])]
    if len(temp_df) == 0:
        print(f'No matching WCM ID for {cornell_subject[12:]}')
        continue

    subject = temp_df['NYU_ID'].values[0]

    if os.path.exists(os.path.join(dwi_folder,subject + '_0022ms.nii')): # there is a repeat scan
        print(cornell_subject[12:],subject)

        for dt in diffusion_times: #iterate through the other diffusion times registering to lowest
            print('Moving',os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
            moving = ants.image_read(os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
            moving_array = moving.numpy()
            nx,ny,nz,nvols = np.shape(moving_array)

            # Get b0 of the diffusion time and get registration
            moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
            moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
            warped_vol = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg['fwdtransforms'],interpolator='linear') # warp first vol for similarity measures

            # # register full dwi - need this for fasciculation measurements
            # registered_dwi = np.zeros([nx,ny,nz,nvols])
            # for vol in range(0,nvols):
            #     temp_array = moving_array[:,:,:,vol]
            #     temp_ants = ants.from_numpy(temp_array, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            #     warped_image = ants.apply_transforms(fixed=fixed_ants,moving=temp_ants,transformlist=reg['fwdtransforms'],interpolator='linear')
            #     warped_image_array = warped_image.numpy()
            #     registered_dwi[:,:,:,vol] = warped_image_array
            # registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
            # save_filename = os.path.join(dwi_folder,subject + '_' + dt + '_reg.nii')
            # registered_dwi_ants.to_filename(save_filename)
            
            # Register maps using global reg
            # maps = ['AD','RD','FA','L2','L3']
            maps = ['AD','RD','FA']
            for map in maps:
                current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
                map_array = current_map.numpy()
                print(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

                warped_map = ants.apply_transforms(
                    fixed=fixed_ants,
                    moving=current_map,
                    transformlist=reg['fwdtransforms'],
                    interpolator='bSpline'  
                )
                save_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
                warped_map.to_filename(save_filename)

            # Now iterate through ROIs to refine
            for label in range(1,5):
                temp_roi = (roi == label)
                temp_roi_name = rois[label-1]
                print(temp_roi_name)

                coords = np.argwhere(temp_roi) # where is there ROI
                pad = 10
                mins = np.maximum(coords.min(0) - pad, 0)
                maxs = np.minimum(coords.max(0) + pad + 1, temp_roi.shape)

                # create bounding box mask
                bbox_mask = np.zeros_like(temp_roi, dtype=np.uint8)
                bbox_mask[mins[0]:maxs[0], mins[1]:maxs[1], mins[2]:maxs[2]] = 1

                bbox_mask_ants = ants.from_numpy(bbox_mask.astype(np.uint8),origin=fixed_ants.origin,spacing=fixed_ants.spacing,direction=fixed_ants.direction)
                bbox_mask_ants = bbox_mask_ants.clone('float')
                similarity_measure = 'MattesMutualInformation'
                # similarity_val = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=moving_ants*bbox_mask_ants,metric_type=similarity_measure) # calculate similarity just in bounding box before warp
                similarity_val = ants.image_similarity(fixed_image=fixed_ants,moving_image=moving_ants,metric_type=similarity_measure,fixed_mask=bbox_mask_ants) # calculate similarity just in bounding box before warp
                print(dt,"Initial Global:", similarity_val) # before first warp
                similarity_val_initial = ants.image_similarity(fixed_image=fixed_ants,moving_image=warped_vol,metric_type=similarity_measure,fixed_mask=bbox_mask_ants) # calculate similarity just in bounding box
                print(dt,"Global after affine:", similarity_val_initial) # after first warp

                # Second reg- rigid just in bounding box around ROI
                reg_roi = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Rigid',initial_transform=reg['fwdtransforms'], mask=bbox_mask_ants,random_seed=42,mask_all_stages=True) #includes intial reg
                warped_vol_roi = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg_roi['fwdtransforms'],interpolator='linear') # roi warp
                similarity_val_roi = ants.image_similarity(fixed_image=fixed_ants,moving_image=warped_vol_roi,metric_type=similarity_measure,fixed_mask=bbox_mask_ants)
                print(dt,"After ROI warp:", similarity_val_roi)

                # apply to maps
                # maps = ['AD','RD','FA','L2','L3']
                maps = ['AD','RD']
                for map in maps:
                    current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
                    map_array = current_map.numpy()

                    if abs(similarity_val_roi) < abs(similarity_val_initial): # it's worse, so just use original warp
                        print('y')
                        global_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
                        warped_map = ants.image_read(global_filename)
                        save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
                        warped_map.to_filename(save_filename)

                    else:
                        warped_map = ants.apply_transforms(
                            fixed=fixed_ants,
                            moving=current_map,
                            transformlist=reg_roi['fwdtransforms'],
                            interpolator='bSpline'  
                        )
                        save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
                        warped_map.to_filename(save_filename)

                temp_data = [cornell_subject,subject,temp_roi_name,similarity_val,similarity_val_initial,similarity_val_roi]
                temp_df = pd.DataFrame([temp_data],columns = column_names)
                df = pd.concat([df,temp_df])
                df.to_csv(csv_filename,index=False)

            # maps = ['eig']
            # for map in maps:
            #     current_map = ants.image_read(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            #     map_array = current_map.numpy()
            #     print(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

            #     warped_map = ants.apply_transforms(
            #         fixed=fixed_ants,
            #         moving=current_map,
            #         transformlist=reg['fwdtransforms'],
            #         interpolator='bSpline',
            #         imagetype=1  
            #     )
            #     save_filename = os.path.join(os.path.join(parent_folder,subject + '_' + dt + '_' + map + '_withinscanreg.nii.gz'))
            #     warped_map.to_filename(save_filename)


No matching WCM ID for 2001
No matching WCM ID for 2002
No matching WCM ID for 2003
No matching WCM ID for 2004
No matching WCM ID for 2005
2006 C2_001
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi_smoothed/C2_001_0022ms.nii
/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed/C2_001_0022ms_AD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed/C2_001_0022ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed/C2_001_0022ms_FA.nii.gz
left_masseter
0022ms Initial Global: -0.3442758619785309
0022ms Global after affine: -0.6693859696388245
0022ms After ROI warp: -0.6664180755615234
y
y
right_masseter
0022ms Initial Global: -0.3378522992134094


/var/folders/rl/51t1ybms78ggvyh_9h7rm87h0000gr/T/ipykernel_35974/2019433073.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,temp_df])


0022ms Global after affine: -0.6606581807136536
0022ms After ROI warp: -0.3478374779224396
y
y
left_temporalis
0022ms Initial Global: -0.5025925636291504
0022ms Global after affine: -0.9140139818191528
0022ms After ROI warp: -0.9058125019073486
y
y
right_temporalis
0022ms Initial Global: -0.531626284122467
0022ms Global after affine: -1.0092253684997559
0022ms After ROI warp: -1.0257587432861328
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi_smoothed/C2_001_0042ms.nii
/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed/C2_001_0042ms_AD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed/C2_001_0042ms_RD.nii.gz
/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed/C2_001_0042ms_FA.nii.gz
left_masseter
0042ms Initial Global: -0.36278432607650757
0042ms Global after affine: -0.6806521415710449
0042ms After ROI warp: -0.6281964182853699
y
y
right_masseter
0042ms Initial Global: -0.33070608973503113
0042ms Global after affin

In [6]:
# HEAL registration maps to lowest diffusion time image- register maps WITHIN scan - batch NOT USING ROIS but fanicer- NYU rescan to Cornell - USE THIS
# Gabrielle Baxter 05/18/2026 gabrielle.baxter@nyulangone.org

import ants
import os
import numpy as np
import pandas as pd

# all_folders = '/Users/gabriellebaxter/Documents/HEAL_Volunteers'
# subjects = [s for s in os.listdir(all_folders) if 'C2_' in s]
# subjects = [s for s in os.listdir(all_folders) if 'WCMyofascial2' in s]
maps_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/maps_smoothed'
dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi_smoothed'
dwi_unsmoothed_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi'
original_dwi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/dwi_smoothed'
roi_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/cornell/roi_resampled'
all_maps = os.listdir(maps_folder)
cornell_subjects = [s[0:16] for s in os.listdir(roi_folder)]
cornell_subjects = list(set(cornell_subjects))
cornell_subjects.sort()

diffusion_times = ['0022ms','0042ms','0081ms','0156ms','0300ms']
rois = ['left_masseter','right_masseter','left_temporalis','right_temporalis']

column_names = ['WCMID','NYUID','ROI','initial_mmi','after_global_mmi','after_roi_mmi']
df = pd.DataFrame(columns = column_names)
csv_filename = 'nyu_mmi.csv'

for cornell_subject in cornell_subjects:
    # Load ROI
    roi_file = os.path.join(roi_folder,cornell_subject + '.nii.gz')
    roi_ants = ants.image_read(roi_file)
    roi = roi_ants.numpy()

    # Get correpsonding map files
    map_files = [s for s in all_maps if cornell_subject in s]
    # print(map_files)

    # Get fixed lowest diffusion time b0 to register everything to
    fixed = ants.image_read(os.path.join(original_dwi_folder,cornell_subject + '_' + diffusion_times[0] + '.nii'))
    fixed_array = fixed.numpy()
    fixed_b0= fixed_array[:,:,:,0]
    fixed_ants = ants.from_numpy(fixed_b0, origin=fixed.origin[:3],spacing=fixed.spacing[:3], direction=fixed.direction[:3, :3])

    matching_codes = pd.read_csv('/Users/gabriellebaxter/Documents/code/HEAL_scripts/rescan_rpbm/matching_codes.csv')
    temp_df = matching_codes[matching_codes['WCM_ID'] == int(cornell_subject[12:])]
    if len(temp_df) == 0:
        print(f'No matching WCM ID for {cornell_subject[12:]}')
        continue

    subject = temp_df['NYU_ID'].values[0]

    if os.path.exists(os.path.join(dwi_folder,subject + '_0022ms.nii')): # there is a repeat scan
        print(cornell_subject[12:],subject)

        for dt in diffusion_times: #iterate through the other diffusion times registering to lowest
            # print('Moving',os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
            # moving = ants.image_read(os.path.join(dwi_folder,subject + '_' + dt + '.nii'))
            # moving_array = moving.numpy()
            # nx,ny,nz,nvols = np.shape(moving_array)

            # # Get b0 of the diffusion time and get registration
            # moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
            # moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            # reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
            # warped_vol = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg['fwdtransforms'],interpolator='linear') # warp first vol for similarity measures

            print('Moving',os.path.join(dwi_unsmoothed_folder,subject + '_' + dt + '.nii'))
            moving = ants.image_read(os.path.join(dwi_unsmoothed_folder,subject + '_' + dt + '.nii'))
            moving_array = moving.numpy()
            nx,ny,nz,nvols = np.shape(moving_array)

            # Get b0 of the diffusion time and get registration
            moving_b0 = moving_array[:,:,:,0] # register maps using b0 warp (eddy registers all volumes to the b0 so maps are in b0 space)
            moving_ants = ants.from_numpy(moving_b0, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
            reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
            warped_vol = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg['fwdtransforms'],interpolator='linear') # warp first vol for similarity measures

            # register full dwi - need this for fasciculation measurements
            registered_dwi = np.zeros([nx,ny,nz,nvols])
            for vol in range(0,nvols):
                temp_array = moving_array[:,:,:,vol]
                temp_ants = ants.from_numpy(temp_array, origin=moving.origin[:3],spacing=moving.spacing[:3], direction=moving.direction[:3, :3])
                warped_image = ants.apply_transforms(fixed=fixed_ants,moving=temp_ants,transformlist=reg['fwdtransforms'],interpolator='linear')
                warped_image_array = warped_image.numpy()
                registered_dwi[:,:,:,vol] = warped_image_array
            registered_dwi_ants = ants.from_numpy(registered_dwi, origin=moving.origin,spacing=moving.spacing, direction=moving.direction) # fully registered image is saved as dwi_reg.nii in case you need it
            save_filename = os.path.join(dwi_unsmoothed_folder,subject + '_' + dt + '_reg.nii')
            registered_dwi_ants.to_filename(save_filename)
            
            # Register maps using global reg
            # maps = ['AD','RD','FA','L2','L3']
            # maps = ['AD','RD','FA']
            # maps = ['MD']
            # for map in maps:
            #     current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            #     map_array = current_map.numpy()
            #     print(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))

            #     warped_map = ants.apply_transforms(
            #         fixed=fixed_ants,
            #         moving=current_map,
            #         transformlist=reg['fwdtransforms'],
            #         interpolator='bSpline',
            #         metric='CC'  
            #     )
            #     save_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
            #     warped_map.to_filename(save_filename)

            # # Now iterate through ROIs to refine
            # for label in range(1,5):
            #     temp_roi = (roi == label)
            #     temp_roi_name = rois[label-1]
            #     print(temp_roi_name)

            #     coords = np.argwhere(temp_roi) # where is there ROI
            #     pad = 10
            #     mins = np.maximum(coords.min(0) - pad, 0)
            #     maxs = np.minimum(coords.max(0) + pad + 1, temp_roi.shape)

            #     # create bounding box mask
            #     bbox_mask = np.zeros_like(temp_roi, dtype=np.uint8)
            #     bbox_mask[mins[0]:maxs[0], mins[1]:maxs[1], mins[2]:maxs[2]] = 1

            #     bbox_mask_ants = ants.from_numpy(bbox_mask.astype(np.uint8),origin=fixed_ants.origin,spacing=fixed_ants.spacing,direction=fixed_ants.direction)
            #     bbox_mask_ants = bbox_mask_ants.clone('float')
            #     similarity_measure = 'MattesMutualInformation'
            #     # similarity_val = ants.image_similarity(fixed_image=fixed_ants*bbox_mask_ants,moving_image=moving_ants*bbox_mask_ants,metric_type=similarity_measure) # calculate similarity just in bounding box before warp
            #     similarity_val = ants.image_similarity(fixed_image=fixed_ants,moving_image=moving_ants,metric_type=similarity_measure,fixed_mask=bbox_mask_ants) # calculate similarity just in bounding box before warp
            #     print(dt,"Initial Global:", similarity_val) # before first warp
            #     similarity_val_initial = ants.image_similarity(fixed_image=fixed_ants,moving_image=warped_vol,metric_type=similarity_measure,fixed_mask=bbox_mask_ants) # calculate similarity just in bounding box
            #     print(dt,"Global after affine:", similarity_val_initial) # after first warp

            #     # Second reg- rigid just in bounding box around ROI
            #     reg_roi = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Rigid',initial_transform=reg['fwdtransforms'], mask=bbox_mask_ants,random_seed=42,mask_all_stages=True) #includes intial reg
            #     warped_vol_roi = ants.apply_transforms(fixed=fixed_ants,moving=moving_ants,transformlist=reg_roi['fwdtransforms'],interpolator='linear') # roi warp
            #     similarity_val_roi = ants.image_similarity(fixed_image=fixed_ants,moving_image=warped_vol_roi,metric_type=similarity_measure,fixed_mask=bbox_mask_ants)
            #     print(dt,"After ROI warp:", similarity_val_roi)

            #     # apply to maps
            #     # maps = ['AD','RD','FA','L2','L3']
            #     maps = ['AD','RD']
            #     for map in maps:
            #         current_map = ants.image_read(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '.nii.gz'))
            #         map_array = current_map.numpy()

            #         if abs(similarity_val_roi) < abs(similarity_val_initial): # it's worse, so just use original warp
            #             print('y')
            #             global_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
            #             warped_map = ants.image_read(global_filename)
            #             save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
            #             warped_map.to_filename(save_filename)

            #         else:
            #             warped_map = ants.apply_transforms(
            #                 fixed=fixed_ants,
            #                 moving=current_map,
            #                 transformlist=reg_roi['fwdtransforms'],
            #                 interpolator='bSpline'  
            #             )
            #             save_filename = os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_' + temp_roi_name + '.nii.gz')
            #             warped_map.to_filename(save_filename)

            #     temp_data = [cornell_subject,subject,temp_roi_name,similarity_val,similarity_val_initial,similarity_val_roi]
            #     temp_df = pd.DataFrame([temp_data],columns = column_names)
            #     df = pd.concat([df,temp_df])
            #     df.to_csv(csv_filename,index=False)

            # if dt == '0022ms':
            #     # Register sigmas
            #     maps = ['sigma']
            #     sigma_folder = '/Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu_rescan/sigma'
            #     for map in maps:
            #         current_map = ants.image_read(os.path.join(sigma_folder,subject + '_sigma.nii'))
            #         map_array = current_map.numpy()
            #         print(os.path.join(sigma_folder,subject + '_sigma.nii'))

            #         warped_map = ants.apply_transforms(
            #             fixed=fixed_ants,
            #             moving=current_map,
            #             transformlist=reg['fwdtransforms'],
            #             interpolator='bSpline'  
            #         )
            #         save_filename = os.path.join(sigma_folder,subject + '_sigma_reg2wcmscan1.nii')
            #         # save_filename = os.path.join(os.path.join(maps_folder,subject + '_' + dt + '_' + map + '_reg2wcmscan1.nii.gz'))
            #         warped_map.to_filename(save_filename)


No matching WCM ID for 2001
No matching WCM ID for 2002
No matching WCM ID for 2003
No matching WCM ID for 2004
No matching WCM ID for 2005
2006 C2_001
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_001_0022ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_001_0042ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_001_0081ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_001_0156ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_001_0300ms.nii
2007 C2_006
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_006_0022ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_006_0042ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_006_0081ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_006_0156ms.nii
Moving /Users/gabriellebaxter/Documents/HEAL_Volunteers/nyu/dwi/C2_006_0300ms.nii
2008 C2_002
Movi